# DriveGuard - Milestone 4 (part 2): Deep Sequence RUL

LSTM / GRU / 1D-CNN over each drive's raw SMART sequence to predict remaining useful life,
trained with a censoring-aware loss. Metrics: concordance index + RUL MAE - directly
comparable to the classical survival models from part 1.

**Settings:** Add data `driveguard-backblaze-interim`; **Accelerator = GPU**; Internet On;
Secret `GITHUB_TOKEN`. ~30-60 min.

In [ ]:
# 1. Clone repo
import os, subprocess
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
URL = f'https://{token}@github.com/keerthirevanth/driveguard-predictive-maintenance.git'
REPO = '/kaggle/working/driveguard-predictive-maintenance'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', URL, REPO], check=True)
import sys; sys.path.insert(0, f'{REPO}/src')
import torch; print('cloned:', os.path.exists(REPO), '| cuda:', torch.cuda.is_available())

In [ ]:
# 2. Install deps (torch is preinstalled on Kaggle)
!pip install -q polars pyarrow mlflow 2>/dev/null
print('deps installed')

In [ ]:
# 3. Link dataset
import os, glob
os.makedirs(f'{REPO}/data/interim', exist_ok=True)
os.makedirs(f'{REPO}/data/processed', exist_ok=True)
qf = glob.glob('/kaggle/input/**/data_*.parquet', recursive=True)
sf = glob.glob('/kaggle/input/**/drive_summary.parquet', recursive=True)
assert qf and sf, 'dataset not attached?'
def link(src, dst):
    if os.path.lexists(dst): os.remove(dst)
    os.symlink(src, dst)
for f in qf: link(f, f"{REPO}/data/interim/{os.path.basename(f)}")
link(sf[0], f'{REPO}/data/processed/drive_summary.parquet')
print('interim:', sorted(os.listdir(f'{REPO}/data/interim')))

In [ ]:
# 4. Build raw SMART sequence windows (L=30)
from pathlib import Path
from driveguard.config import load_config
from driveguard.features.sequence_data import build_and_save
ROOT = Path(REPO); cfg = load_config(f'{REPO}/config/config.yaml')
info = build_and_save(cfg, ROOT, L=30)
for k in ['train', 'val', 'test']:
    print(k, info[k])

In [ ]:
# 5. Train + evaluate the sequence models
import json
from driveguard.models.sequence_rul import run_sequence_rul
SDIR = f'{REPO}/data/processed/sequences'
board = run_sequence_rul(SDIR, ['lstm', 'gru', 'cnn1d'],
                         mlflow_uri='/kaggle/working/mlruns', epochs=25)
json.dump(board, open('/kaggle/working/sequence_rul_leaderboard.json', 'w'), indent=2)
print('sequence RUL complete')

In [ ]:
# 6. Leaderboard (compare with classical survival C-index)
import pandas as pd
rows = []
for r in board:
    if r.get('status') == 'ok':
        t = r['test']
        rows.append({'model': r['model'], 'c_index': round(t['c_index'], 4),
                     'rul_mae_days': round(t.get('rul_mae_days') or 0, 1),
                     'fit_sec': r.get('fit_sec')})
    else:
        rows.append({'model': r['model'], 'c_index': 'ERROR: ' + r.get('error','')[:60]})
pd.DataFrame(rows).sort_values('c_index', ascending=False)

In [ ]:
# 7. Package artifacts
import shutil, os
if os.path.isdir('/kaggle/working/mlruns'):
    shutil.make_archive('/kaggle/working/mlruns_export', 'zip',
                        root_dir='/kaggle/working', base_dir='mlruns')
!ls -lh /kaggle/working/*.json